# word2vec trên Wikipedia tiếng Việt

Bạn train word2vec skip-gram với negative sampling (SGNS) trên khoảng 6 triệu âm tiết Wikipedia tiếng Việt, tự phát
hiện cụm âm tiết hay đi cùng nhau (`việt nam` thành `việt_nam`), rồi xem láng giềng và phép tính analogy thay đổi ra sao.
Bài Basic 02 đã viết SGNS bằng numpy; ở đây phần tính toán dùng tensor PyTorch cho nhanh, nhưng loss, lấy mẫu âm,
subsampling, cửa sổ động vẫn là của bạn (không dùng gensim).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải `wikipedia-vi` vào `data/raw/wikipedia-vi` nếu chưa có.
- Chấm: `uv run pytest tracks/ai-engineer/modules/04-nlp-sequence-models/exercises/intermediate/01-word2vec-wikipedia -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải Wikipedia tiếng Việt nếu chưa có (117 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy/torch
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

WIKI = ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi" / "train-00001-of-00004.parquet"
print(SHARD, SHARD.exists())

In [ ]:
import re
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Corpus: bài dài, câu, âm tiết

Shard Wikipedia có rất nhiều bài sơ khai do bot tạo (một loài bọ cánh cứng, hai câu). Lấy bài có ít nhất `min_chars` ký
tự rồi chọn ngẫu nhiên `n_articles` bài.

- `load_articles(path, n_articles, min_chars, seed) -> list[str]`: đọc cột `text` của tệp parquet, giữ bài có
  `len(text) >= min_chars`, lấy mẫu bằng `DataFrame.sample(n=n_articles, random_state=seed)` (đúng hàm này để ra cùng
  bộ bài với lời giải), trả danh sách văn bản theo thứ tự mẫu.
- `tokenize(text) -> list[list[str]]`: chuẩn hoá **NFC** rồi `lower()`, cắt câu bằng `SENT_RE` (dấu `.!?;:` và xuống
  dòng), mỗi câu lấy các âm tiết bằng `TOKEN_RE` (chỉ chữ cái, bỏ số và `_`), bỏ câu có dưới 2 âm tiết.

In [ ]:
SENT_RE = re.compile(r"[.!?;:\n]+")
TOKEN_RE = re.compile(r"[^\W\d_]+")


def load_articles(path, n_articles: int, min_chars: int, seed: int) -> list[str]:
    df = pd.read_parquet(path, columns=["text"])
    df = df[df.text.str.len() >= min_chars]
    return df.sample(n=n_articles, random_state=seed).text.tolist()


def tokenize(text: str) -> list[list[str]]:
    text = unicodedata.normalize("NFC", text).lower()
    sentences = []
    for part in SENT_RE.split(text):
        tokens = TOKEN_RE.findall(part)
        if len(tokens) >= 2:
            sentences.append(tokens)
    return sentences

In [ ]:
# explore: 5 000 bài dài lấy ngẫu nhiên (seed 0), khoảng 6.4 triệu âm tiết
start = time.perf_counter()
articles = load_articles(SHARD, n_articles=5000, min_chars=2000, seed=0)
sentences = [s for a in articles for s in tokenize(a)]
n_tokens = sum(map(len, sentences))
print(f"{len(sentences)} câu, {n_tokens} âm tiết, {time.perf_counter() - start:.1f} s")
print(sentences[0])

## 2. Ghép âm tiết thành cụm

Điểm của cặp âm tiết liền nhau $$(a, b)$$, với $$N$$ là tổng số âm tiết của corpus:

$$
\text{score}(a, b) = \frac{(\text{count}(ab) - \delta) \cdot N}{\text{count}(a)\,\text{count}(b)}
$$

- `find_phrases(sentences, min_count, delta, threshold) -> set[tuple[str, str]]`: đếm âm tiết và cặp liền nhau **trong
  từng câu** (không ghép qua ranh giới câu), giữ cặp có `count(ab) >= min_count` và điểm **lớn hơn hẳn** `threshold`.
- `apply_phrases(sentence, phrases) -> list[str]`: đi từ trái sang phải; nếu âm tiết hiện tại và âm tiết kế tiếp là một
  cụm thì nối bằng `_` rồi nhảy qua cả hai, ngược lại giữ nguyên. Một lượt, không ghép tiếp cụm đã ghép.

In [ ]:
def find_phrases(sentences: list[list[str]], min_count: int, delta: float, threshold: float) -> set[tuple[str, str]]:
    unigram = Counter(w for s in sentences for w in s)
    bigram = Counter(p for s in sentences for p in zip(s, s[1:]))
    n = sum(unigram.values())
    return {
        (a, b) for (a, b), c in bigram.items()
        if c >= min_count and (c - delta) * n / (unigram[a] * unigram[b]) > threshold
    }


def apply_phrases(sentence: list[str], phrases: set[tuple[str, str]]) -> list[str]:
    out, i = [], 0
    while i < len(sentence):
        if i + 1 < len(sentence) and (sentence[i], sentence[i + 1]) in phrases:
            out.append(sentence[i] + "_" + sentence[i + 1])
            i += 2
        else:
            out.append(sentence[i])
            i += 1
    return out

In [ ]:
# explore: phát hiện cụm, xem cụm phổ biến nhất
phrases = find_phrases(sentences, min_count=5, delta=5, threshold=100)
bigram = Counter(p for s in sentences for p in zip(s, s[1:]))
print(len(phrases), "cụm, ví dụ:", ["_".join(p) for p in sorted(phrases, key=lambda p: -bigram[p])[:20]])
print("trung quốc là cụm?", ("trung", "quốc") in phrases, "| việt nam?", ("việt", "nam") in phrases)
phrased = [apply_phrases(s, phrases) for s in sentences]
print("token sau khi ghép:", sum(map(len, phrased)))

## 3. Vocab, subsampling, cặp skip-gram

Corpus được trải thành hai mảng song song: `ids` (id của từng token, theo thứ tự) và `sent_ids` (token đó thuộc câu
nào). Cặp skip-gram chỉ lấy trong cùng một câu.

- `build_vocab(sentences, min_count) -> (words, counts)`: từ xuất hiện `>= min_count` lần, xếp theo tần suất giảm dần,
  hoà thì theo thứ tự chữ cái; `counts` là `np.int64`.
- `flatten(sentences, word2id) -> (ids, sent_ids)`: bỏ token ngoài vocab; hai mảng `np.int64` cùng độ dài.
- `keep_probs(counts, t)`: như Basic 02, $$\min\left(1, \left(\sqrt{f/t} + 1\right) t / f\right)$$.
- `make_pairs(ids, sent_ids, window, keep, rng) -> np.ndarray (P, 2)`, theo đúng thứ tự gọi `rng`:
  1. Subsample: `mask = rng.random(len(ids)) < keep[ids]`, giữ token có `mask` đúng.
  2. Cửa sổ động: `b = rng.integers(1, window + 1, size=số token còn lại)`, một số cho mỗi token **làm center**.
  3. Với mỗi khoảng cách `d` từ 1 tới `window`, sinh cặp `(center, context)` cho cả hai phía (context ở `i + d` và
     `i - d`) khi hai token cùng câu và `d <= b[center]`. Thứ tự các cặp không quan trọng (sẽ xáo). Dùng numpy
     vectorized, đừng lặp Python từng token: corpus có hàng triệu token.

In [ ]:
def build_vocab(sentences: list[list[str]], min_count: int) -> tuple[list[str], np.ndarray]:
    freq = Counter(w for s in sentences for w in s)
    words = sorted((w for w, c in freq.items() if c >= min_count), key=lambda w: (-freq[w], w))
    return words, np.array([freq[w] for w in words], dtype=np.int64)


def flatten(sentences: list[list[str]], word2id: dict[str, int]) -> tuple[np.ndarray, np.ndarray]:
    ids, sent_ids = [], []
    for k, s in enumerate(sentences):
        for w in s:
            if w in word2id:
                ids.append(word2id[w])
                sent_ids.append(k)
    return np.array(ids, dtype=np.int64), np.array(sent_ids, dtype=np.int64)


def keep_probs(counts: np.ndarray, t: float = 1e-4) -> np.ndarray:
    f = counts / counts.sum()
    return np.minimum(1.0, (np.sqrt(f / t) + 1) * t / f)


def make_pairs(ids: np.ndarray, sent_ids: np.ndarray, window: int, keep: np.ndarray,
               rng: np.random.Generator) -> np.ndarray:
    mask = rng.random(len(ids)) < keep[ids]
    a, s = ids[mask], sent_ids[mask]
    b = rng.integers(1, window + 1, size=len(a))
    pairs = []
    for d in range(1, window + 1):
        same = s[:-d] == s[d:]
        right = same & (b[:-d] >= d)                     # center ở i, context ở i + d
        left = same & (b[d:] >= d)                       # center ở i + d, context ở i
        pairs.append(np.stack([a[:-d][right], a[d:][right]], axis=1))
        pairs.append(np.stack([a[d:][left], a[:-d][left]], axis=1))
    return np.concatenate(pairs).astype(np.int64).reshape(-1, 2)

## 4. SGNS bằng tensor PyTorch

- `SGNS(vocab_size, dim)`: hai `nn.Embedding(vocab_size, dim, sparse=True)` tên `in_emb` và `out_emb`. `in_emb` khởi tạo
  đều trong $$[-0.5/d, 0.5/d]$$, `out_emb` bằng 0 (như mã C gốc).
  `forward(centers, contexts, negatives)` trả loss **trung bình trên batch**:
  $$-\log\sigma(u_o^\top v_c) - \sum_k \log\sigma(-u_k^\top v_c)$$. Dùng `F.logsigmoid`, đừng `log(sigmoid(...))`.
- `train_sgns(ids, sent_ids, counts, dim, window, negatives, epochs, batch_size, lr, seed, sample=1e-4) -> np.ndarray`:
  - `torch.manual_seed(seed)` rồi tạo `SGNS`; `rng = np.random.default_rng(seed)`.
  - Mỗi epoch: `make_pairs(..., keep_probs(counts, sample), rng)`, xáo bằng `rng.permutation`, đi theo batch.
  - Mẫu âm: `torch.multinomial(noise, B * K, replacement=True)` với `noise` là $$\text{count}^{0.75}$$ chuẩn hoá.
  - `torch.optim.SGD` trên tham số sparse. word2vec gốc dùng lr **mỗi cặp** 0.025 giảm tuyến tính về 0; loss của ta là
    trung bình nên lr của optimizer là `lr * batch_size`, giảm tuyến tính theo bước tới gần 0 (cộng sàn `1e-4 * batch_size`).
  - In loss trung bình mỗi epoch; trả `in_emb.weight` dạng numpy `(V, dim)`.

In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, dim, sparse=True)
        self.out_emb = nn.Embedding(vocab_size, dim, sparse=True)
        with torch.no_grad():
            self.in_emb.weight.uniform_(-0.5 / dim, 0.5 / dim)
            self.out_emb.weight.zero_()

    def forward(self, centers: torch.Tensor, contexts: torch.Tensor, negatives: torch.Tensor) -> torch.Tensor:
        v = self.in_emb(centers)                                   # (B, d)
        pos = (v * self.out_emb(contexts)).sum(-1)                 # (B,)
        neg = torch.bmm(self.out_emb(negatives), v.unsqueeze(-1)).squeeze(-1)   # (B, K)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(-1)).mean()


def train_sgns(ids: np.ndarray, sent_ids: np.ndarray, counts: np.ndarray, dim: int, window: int, negatives: int,
               epochs: int, batch_size: int, lr: float, seed: int, sample: float = 1e-4) -> np.ndarray:
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = SGNS(len(counts), dim)
    noise = torch.tensor(counts ** 0.75 / (counts ** 0.75).sum())
    keep = keep_probs(counts, sample)
    opt = torch.optim.SGD(model.parameters(), lr=lr * batch_size)
    pairs_per_epoch = None
    step = 0
    for epoch in range(epochs):
        pairs = make_pairs(ids, sent_ids, window, keep, rng)
        pairs = torch.from_numpy(pairs[rng.permutation(len(pairs))])
        if pairs_per_epoch is None:
            pairs_per_epoch = len(pairs)
            total_steps = epochs * -(-pairs_per_epoch // batch_size)
        losses = []
        for start in range(0, len(pairs), batch_size):
            frac = min(step / total_steps, 1.0)
            for group in opt.param_groups:
                group["lr"] = (lr * (1 - frac) + 1e-4) * batch_size
            batch = pairs[start:start + batch_size]
            neg = torch.multinomial(noise, len(batch) * negatives, replacement=True).view(len(batch), negatives)
            loss = model(batch[:, 0], batch[:, 1], neg)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
            step += 1
        print(f"epoch {epoch + 1}: {len(pairs)} cặp, loss {np.mean(losses):.4f}")
    return model.in_emb.weight.detach().numpy().copy()

## 5. Láng giềng và analogy

- `most_similar(E, words, word, k) -> list[tuple[str, float]]`: `k` từ có cosine cao nhất với `word`, không tính chính
  nó, giảm dần.
- `analogy(E, words, a, b, c, k) -> list[str]`: "a với b như c với ?": chuẩn hoá mọi vector về độ dài 1, tính
  $$\hat v_b - \hat v_a + \hat v_c$$, trả `k` từ có cosine cao nhất, **bỏ** `a`, `b`, `c`.
- `analogy_accuracy(E, words, pairs) -> tuple[int, int]`: với mọi cặp có thứ tự `(a, b)`, `(c, d)` khác nhau trong
  `pairs` mà cả 4 từ đều có trong vocab, đếm số lần `analogy(E, words, a, b, c, 1)[0] == d`. Trả `(đúng, tổng)`.

In [ ]:
def _unit(E: np.ndarray) -> np.ndarray:
    return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)


def most_similar(E: np.ndarray, words: list[str], word: str, k: int) -> list[tuple[str, float]]:
    index = {w: i for i, w in enumerate(words)}
    U = _unit(E)
    sims = U @ U[index[word]]
    sims[index[word]] = -np.inf
    top = np.argsort(-sims, kind="stable")[:k]
    return [(words[i], float(sims[i])) for i in top]


def analogy(E: np.ndarray, words: list[str], a: str, b: str, c: str, k: int) -> list[str]:
    index = {w: i for i, w in enumerate(words)}
    U = _unit(E)
    sims = U @ (U[index[b]] - U[index[a]] + U[index[c]])
    for w in (a, b, c):
        sims[index[w]] = -np.inf
    return [words[i] for i in np.argsort(-sims, kind="stable")[:k]]


def analogy_accuracy(E: np.ndarray, words: list[str], pairs: list[tuple[str, str]]) -> tuple[int, int]:
    vocab = set(words)
    pairs = [p for p in pairs if p[0] in vocab and p[1] in vocab]
    correct = total = 0
    for a, b in pairs:
        for c, d in pairs:
            if (a, b) == (c, d):
                continue
            total += 1
            correct += analogy(E, words, a, b, c, 1)[0] == d
    return correct, total

In [ ]:
# explore: train hai lần, trên âm tiết và sau khi ghép cụm (mỗi lần khoảng 1-2 phút trên CPU 4 luồng)
torch.set_num_threads(4)
models = {}
for name, corpus in [("âm tiết", sentences), ("ghép cụm", phrased)]:
    words, counts = build_vocab(corpus, min_count=5)
    ids, sent_ids = flatten(corpus, {w: i for i, w in enumerate(words)})
    print(f"--- {name}: vocab {len(words)}, {len(ids)} token")
    start = time.perf_counter()
    E = train_sgns(ids, sent_ids, counts, dim=100, window=5, negatives=5, epochs=3, batch_size=1024, lr=0.025, seed=0)
    print(f"{time.perf_counter() - start:.0f} s")
    models[name] = (E, words)

In [ ]:
# explore: láng giềng gần nhất
for name, (E, words) in models.items():
    print("===", name)
    vocab = set(words)
    for w in ["hà_nội", "hà", "bóng_đá", "mèo", "sông", "đỏ"]:
        if w in vocab:
            print(f"{w:9}", ", ".join(f"{x} {s:.2f}" for x, s in most_similar(E, words, w, 6)))
        else:
            print(f"{w:9}", "không có trong vocab")

In [ ]:
# explore: analogy nước - thủ đô và giới tính (chỉ tính cặp có trong vocab)
CAPITALS = [("việt_nam", "hà_nội"), ("nhật_bản", "tokyo"), ("trung_quốc", "bắc_kinh"), ("pháp", "paris"),
            ("đức", "berlin"), ("nga", "moskva"), ("anh", "luân_đôn"), ("hàn_quốc", "seoul"), ("thái_lan", "bangkok"),
            ("ý", "roma"), ("tây_ban_nha", "madrid"), ("ai_cập", "cairo"), ("hoa_kỳ", "washington"), ("áo", "viên"),
            ("hy_lạp", "athena"), ("lào", "viêng_chăn"), ("campuchia", "phnôm_pênh"), ("ấn_độ", "new_delhi"),
            ("triều_tiên", "bình_nhưỡng"), ("thổ_nhĩ_kỳ", "ankara")]
GENDER = [("ông", "bà"), ("cha", "mẹ"), ("chồng", "vợ"), ("vua", "hoàng_hậu"), ("hoàng_tử", "công_chúa"),
          ("con_trai", "con_gái"), ("anh_trai", "chị_gái"), ("nam", "nữ"), ("bố", "mẹ"), ("chú", "cô"),
          ("hoàng_đế", "hoàng_hậu"), ("đàn_ông", "phụ_nữ")]
E, words = models["ghép cụm"]
for name, pairs in [("nước-thủ đô", CAPITALS), ("giới tính", GENDER)]:
    ok, total = analogy_accuracy(E, words, pairs)
    print(f"{name}: {ok}/{total} = {ok / total:.3f}")
print("pháp : paris :: đức : ?", analogy(E, words, "pháp", "paris", "đức", 5))

In [ ]:
# explore: chiếu PCA 2 chiều các nước và thủ đô
import matplotlib.pyplot as plt

index = {w: i for i, w in enumerate(words)}
pts = [w for p in CAPITALS for w in p if p[0] in index and p[1] in index]
X = E[[index[w] for w in pts]]
X = X - X.mean(0)
_, _, Vt = np.linalg.svd(X, full_matrices=False)
P = X @ Vt[:2].T
plt.figure(figsize=(8, 6))
plt.scatter(P[:, 0], P[:, 1], s=10)
for (x, y), w in zip(P, pts):
    plt.annotate(w, (x, y), fontsize=8)
for i in range(0, len(pts), 2):
    plt.plot(P[i:i + 2, 0], P[i:i + 2, 1], color="gray", linewidth=0.5)
plt.title("Nước và thủ đô (PCA của vector word2vec)")
plt.show()

## 6. Đọc kết quả

So với bảng ở mục 3 phần lý thuyết: không ghép cụm thì `hà_nội` không tồn tại và láng giềng của `hà`, `mèo` là các
**âm tiết** trong tên riêng, từ ghép; ghép cụm thì láng giềng có nghĩa hơn hẳn. Analogy chỉ đúng khoảng 10-20%: 6 triệu
token là quá ít cho quan hệ tuyến tính (word2vec gốc dùng hàng tỉ từ). Số của bạn có thể lệch nhẹ so với lý thuyết nếu số
luồng khác (thứ tự cộng float32), nhưng xu hướng phải giống.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.